# SI4006 · Entrega M1 — Fine-tuning con LoRA (Encoder-Decoder)

**Tarea:** transformar diálogos médico-paciente (`dialogue`) en notas clínicas (`section_text`), usando el dataset MTS-Dialog.

---

Este notebook está basado en el lab de la semana 4 (`S04_Lab_Fine-tuning_LoRA.ipynb`), pero adaptado a un modelo
encoder-decoder. El input y el output son ambos texto libre, así que usamos un
modelo Seq2Seq (Flan-T5) en vez de un encoder de clasificación.

## 0 · Setup

Igual que en el lab: no fijamos `transformers`/`torch` (usamos los de Colab), solo instalamos lo que falta.
Agregamos `rouge_score` y `bert_score` porque son nuestras métricas principales (por ser una tarea de generación de texto y no clasificación como en el lad 4).

In [1]:
# Instalamos SOLO lo que falta. No fijamos transformers/torch (usamos los de Colab).
%pip install -q peft datasets evaluate accelerate rouge_score bert_score
# Quitamos el torchao viejo de Colab (choca con peft en get_peft_model; no lo usamos aquí).
%pip uninstall -y torchao
print('\nListo.')

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.2 MB/s eta 0:00:00
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0

Listo.


In [2]:
import torch, transformers, peft
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('transformers', transformers.__version__, '| peft', peft.__version__)
print('torch', torch.__version__, '| device:', device)
if device == 'cpu':
    print('\n⚠️  Estás en CPU. Funciona, pero entrena lento. Activa la GPU T4 (ver arriba).')

transformers 5.16.1 | peft 0.20.0
torch 2.11.0+cu128 | device: cuda


## 1 · Los datos: MTS-Dialog

El dataset viene repartido en 4 archivos CSV en GitHub, usamos directamente la partición que el dataset ya trae:

- `MTS-Dialog-TrainingSet.csv` corresponde a  train
- `MTS-Dialog-ValidationSet.csv` corresponde a validation
- `MTS-Dialog-TestSet-1-MEDIQA-Chat-2023.csv` + `MTS-Dialog-TestSet-2-MEDIQA-Sum-2023.csv` los unimos en test
  (son dos sets de test con el mismo objetivo).

Cada fila tiene `ID`, `section_header`, `section_text` y `dialogue`. Nuestra tarea usa `dialogue` como input y
`section_text` como output; `section_header` no nos interesa y lo descartamos.

El resto de la descripción del dataset está en el markdown específico de la entrega para esto.

In [3]:
import pandas as pd

# Dataset
url_train = 'https://raw.githubusercontent.com/abachaa/MTS-Dialog/refs/heads/main/Main-Dataset/MTS-Dialog-TrainingSet.csv'
url_val   = 'https://raw.githubusercontent.com/abachaa/MTS-Dialog/refs/heads/main/Main-Dataset/MTS-Dialog-ValidationSet.csv'
url_test1 = 'https://raw.githubusercontent.com/abachaa/MTS-Dialog/refs/heads/main/Main-Dataset/MTS-Dialog-TestSet-1-MEDIQA-Chat-2023.csv'
url_test2 = 'https://raw.githubusercontent.com/abachaa/MTS-Dialog/refs/heads/main/Main-Dataset/MTS-Dialog-TestSet-2-MEDIQA-Sum-2023.csv'

cols = ['dialogue', 'section_text']

train_df = pd.read_csv(url_train)[cols].dropna().reset_index(drop=True)
val_df   = pd.read_csv(url_val)[cols].dropna().reset_index(drop=True)
test_df  = pd.concat([pd.read_csv(url_test1)[cols], pd.read_csv(url_test2)[cols]], ignore_index=True).dropna()
test_df  = test_df.reset_index(drop=True)

print('train:', train_df.shape)
print('validation:', val_df.shape)
print('test:', test_df.shape)

print('\nUn ejemplo real:')
print('  dialogue      :', train_df.loc[0, 'dialogue'][:200], '...')
print('  section_text  :', train_df.loc[0, 'section_text'][:200], '...')

train: (1201, 2)
validation: (100, 2)
test: (400, 2)

Un ejemplo real:
  dialogue      : Doctor: What brings you back into the clinic today, miss? 
Patient: I came in for a refill of my blood pressure medicine. 
Doctor: It looks like Doctor Kumar followed up with you last time regarding ...
  section_text  : The patient is a 76-year-old white female who presents to the clinic today originally for hypertension and a med check.  She has a history of hypertension, osteoarthritis, osteoporosis, hypothyroidism ...


## 2 · Modelo base + tokenizer, y tokenización

Cargamos Flan-T5-base (~250M, que es un encoder-decoder) con su tokenizer. Elegimos Flan-T5 en vez de un
T5 crudo porque, al estar afinado con instrucciones (instruction tuning), su baseline zero-shot (sin nuestro fine-tuning) ya produce
algo razonable cuando le damos una instrucción en texto, que es el objetivo del instruction tuning. Esto permite que el baseline es un punto de comparación honesta y no
solo ruido. Sigue siendo lo suficientemente pequeño para ser ejecutado en Colab.

Usamos el mismo prompt en el baseline y en el fine-tuning para que la comparación sea justa.

In [4]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODELO = 'google/flan-t5-base'
tokenizer = AutoTokenizer.from_pretrained(MODELO)
model = AutoModelForSeq2SeqLM.from_pretrained(MODELO)
model.to(device)
model.generation_config.no_repeat_ngram_size = 4  # bloquea repetir cualquier secuencia de 4 tokens ya generada


PROMPT = 'Summarize the following doctor-patient dialogue into a clinical note:\n\n{dialogue}'

MAX_INPUT_LEN  = 512   # cubre p95 de la longitud de los diálogos (train)
MAX_TARGET_LEN = 200   # cubre p95 de la longitud de las notas clínicas de referencia

def tokenizar(batch):
    inputs = [PROMPT.format(dialogue=d) for d in batch['dialogue']]
    model_inputs = tokenizer(inputs, truncation=True, max_length=MAX_INPUT_LEN)
    labels = tokenizer(text_target=batch['section_text'], truncation=True, max_length=MAX_TARGET_LEN)
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

print('Modelo y tokenizer listos.')

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Modelo y tokenizer listos.


In [5]:
from datasets import Dataset

train_ds = Dataset.from_pandas(train_df).map(tokenizar, batched=True, remove_columns=['dialogue', 'section_text'])
val_ds   = Dataset.from_pandas(val_df).map(tokenizar, batched=True, remove_columns=['dialogue', 'section_text'])
test_ds  = Dataset.from_pandas(test_df).map(tokenizar, batched=True, remove_columns=['dialogue', 'section_text'])

print(train_ds)

Map:   0%|          | 0/1201 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1201
})


## 3 · El baseline: el mismo modelo sin fine-tuning (zero-shot)

El baseline es el modelo base sin fine-tuning, usando el mismo prompt que
se usará con el modelo tuneado. Medimos ROUGE y BERTScore sobre todo el conjunto de validación, para poder comparar
contra el modelo afinado sobre exactamente el mismo conjunto más adelante.

In [6]:
import evaluate
from tqdm import tqdm

rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")
model.eval()

baseline_df = val_df.copy()
references = baseline_df["section_text"].tolist()

baseline_predictions = []
BATCH = 8
dialogues = baseline_df["dialogue"].tolist()

for i in tqdm(range(0, len(dialogues), BATCH), desc="Generating baseline"):
    prompts = [PROMPT.format(dialogue=d) for d in dialogues[i:i + BATCH]]
    inp = tokenizer(prompts, return_tensors="pt", truncation=True,
                    max_length=MAX_INPUT_LEN, padding=True).to(device)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=200, num_beams=4)
    baseline_predictions.extend(tokenizer.batch_decode(out, skip_special_tokens=True))

rouge_results = rouge.compute(predictions=baseline_predictions, references=references)
bert_results = bertscore.compute(predictions=baseline_predictions, references=references,
                                 lang="en", rescale_with_baseline=True)
bert_f1 = sum(bert_results["f1"]) / len(bert_results["f1"])
print(f"\nBASELINE — BERTScore\nF1: {bert_f1:.4f}")

print("BASELINE — ROUGE")
for metric, score in rouge_results.items():
    print(f"{metric}: {score:.4f}")

metrics = {
    **rouge_results,
    "bert_f1": bert_f1
}

pd.DataFrame([metrics]).to_csv("baseline_metrics.csv", index=False)

Generating baseline: 100%|██████████| 13/13 [00:50<00:00,  3.91s/it]


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



BASELINE — BERTScore
F1: 0.2964
BASELINE — ROUGE
rouge1: 0.2387
rouge2: 0.0900
rougeL: 0.2015
rougeLsum: 0.2012


## 4 · LoRA: fine-tuning eficiente

Igual que en el lab, congelamos el modelo y aprendemos solo las matrices pequeñas de LoRA. Hicimos los siguientes cambios a comparación del notebook de la semana 4:

- `task_type=TaskType.SEQ_2_SEQ_LM` (a diferencia del lab, que usaba `SEQ_CLS` para clasificación).
- `target_modules=['q', 'v']`: en la familia T5, las proyecciones de atención se llaman `q` y `v`
  (no `q_lin`/`v_lin` como en DistilBERT, ni `q_proj`/`v_proj` como en LLaMA/Qwen).
- `r=8`, `lora_alpha=16` (regla común `alpha ≈ 2·r`), igual que el lab y el paper original de LoRA.

In [7]:
from peft import LoraConfig, get_peft_model, TaskType

lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=['q', 'v'],   # capas de atención de T5 / Flan-T5
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()   # miren el % de parámetros que se entrena

trainable params: 884,736 || all params: 248,462,592 || trainable%: 0.3561


## 5 · Entrenar con la Seq2SeqTrainer API

Usamos `Seq2SeqTrainer` en vez de `Trainer` porque necesitamos `predict_with_generate=True`. Para tareas de
generación, la métrica se calcula sobre el texto generado, no sobre los logits crudos (a diferencia de
accuracy en el lab, que sí se calculaba directo sobre logits).

In [8]:
import numpy as np
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple):
        preds = preds[0]
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    return rouge.compute(predictions=decoded_preds, references=decoded_labels)

args = Seq2SeqTrainingArguments(
    output_dir='./m1_lora_out',
    learning_rate=2e-4,               # LoRA aguanta un LR más alto que el full fine-tuning
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy='epoch',
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,
    logging_steps=25,
    seed=42,
    report_to='none',
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)
trainer.train()

Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,2.469269,2.225816,0.304883,0.116918,0.239398,0.239242
2,2.443720,2.180749,0.324675,0.119487,0.262260,0.263032
3,2.491585,2.171318,0.327904,0.114022,0.265057,0.266206


TrainOutput(global_step=453, training_loss=2.518582573526479, metrics={'train_runtime': 556.8844, 'train_samples_per_second': 6.47, 'train_steps_per_second': 0.813, 'total_flos': 1853778385198080.0, 'train_loss': 2.518582573526479, 'epoch': 3.0})

## 6 · Evaluar y comparar contra el baseline (validación)

Evaluamos el delta que aportó el fine-tuning sobre exactamente el mismo conjunto de validación que usamos en la Sección 3. Primero computamos las métricas para el modelo con fine-tuning.

In [9]:
rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")
model.eval()

finetuned_df = val_df.copy()
references = finetuned_df["section_text"].tolist()

finetuned_predictions = []
BATCH = 8
dialogues = finetuned_df["dialogue"].tolist()

for i in tqdm(range(0, len(dialogues), BATCH), desc="Generating finetuned"):
    prompts = [PROMPT.format(dialogue=d) for d in dialogues[i:i + BATCH]]
    inp = tokenizer(prompts, return_tensors="pt", truncation=True,
                    max_length=MAX_INPUT_LEN, padding=True).to(device)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=200, num_beams=4)
    finetuned_predictions.extend(tokenizer.batch_decode(out, skip_special_tokens=True))

rouge_results = rouge.compute(predictions=finetuned_predictions, references=references)
bert_results = bertscore.compute(predictions=finetuned_predictions, references=references,
                                 lang="en", rescale_with_baseline=True)
bert_f1 = sum(bert_results["f1"]) / len(bert_results["f1"])

print(f"\nFINETUNED — BERTScore\nF1: {bert_f1:.4f}")

print("FINETUNED — ROUGE")
for metric, score in rouge_results.items():
    print(f"{metric}: {score:.4f}")

metrics = {
    **rouge_results,
    "bert_f1": bert_f1
}

pd.DataFrame([metrics]).to_csv("finetuned_metrics.csv", index=False)

Generating finetuned: 100%|██████████| 13/13 [00:57<00:00,  4.45s/it]


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



FINETUNED — BERTScore
F1: 0.3962
FINETUNED — ROUGE
rouge1: 0.3459
rouge2: 0.1325
rougeL: 0.2836
rougeLsum: 0.2853


A continuación, computamos el delta y comparamos ambos modelos.


In [10]:
baseline = pd.read_csv("baseline_metrics.csv")
finetuned = pd.read_csv("finetuned_metrics.csv")

print("\nMETRICS COMPARISON")

for metric in ["rouge1", "rouge2", "rougeL", "rougeLsum", "bert_f1"]:
    base = baseline[metric].iloc[0]
    fine = finetuned[metric].iloc[0]
    delta = fine - base

    print(f"{metric:10s} | Baseline: {base:.4f} | Finetuned: {fine:.4f} | Δ: {delta:+.4f}")


METRICS COMPARISON
rouge1     | Baseline: 0.2387 | Finetuned: 0.3459 | Δ: +0.1072
rouge2     | Baseline: 0.0900 | Finetuned: 0.1325 | Δ: +0.0425
rougeL     | Baseline: 0.2015 | Finetuned: 0.2836 | Δ: +0.0821
rougeLsum  | Baseline: 0.2012 | Finetuned: 0.2853 | Δ: +0.0841
bert_f1    | Baseline: 0.2964 | Finetuned: 0.3962 | Δ: +0.0998


## 7 · Evaluación final sobre el conjunto de test

El conjunto de test no se tocó ni para entrenar ni para elegir
hiperparámetros, entonces es una medición honesta final del modelo fine-tuneado.

In [11]:
model.eval()

test_predictions = []

dialogues = test_df["dialogue"].tolist()
BATCH = 8

model.eval()

for i in tqdm(range(0, len(dialogues), BATCH), desc="Generating test"):
    prompts = [PROMPT.format(dialogue=d) for d in dialogues[i:i + BATCH]]

    inp = tokenizer(
        prompts,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_LEN,
        padding=True
    ).to(device)

    with torch.no_grad():
        out = model.generate(
            **inp,
            max_new_tokens=200,
            num_beams=4
        )

    test_predictions.extend(
        tokenizer.batch_decode(out, skip_special_tokens=True)
    )

references = test_df["section_text"].tolist()

rouge_test = rouge.compute(
    predictions=test_predictions,
    references=references
)

bert_test = bertscore.compute(
    predictions=test_predictions,
    references=references,
    lang="en",
    rescale_with_baseline=True
)

bert_f1 = sum(bert_test["f1"]) / len(bert_test["f1"])

print("=== FINE-TUNED — TEST (held-out) ===")

print("\nROUGE")
for metrica, valor in rouge_test.items():
    print(f"{metrica}: {valor:.4f}")

print(f"\nBERTScore F1: {bert_f1:.4f}")

Generating test: 100%|██████████| 50/50 [04:25<00:00,  5.31s/it]


=== FINE-TUNED — TEST (held-out) ===

ROUGE
rouge1: 0.3469
rouge2: 0.1591
rougeL: 0.2875
rougeLsum: 0.2875

BERTScore F1: 0.3926


## 8 · Ejemplos cualitativos

Se piden al menos 3 ejemplos pero aquí mostramos 5. Se compara el input que recibe el modelo y se muestra el label que tiene el dataset original y luego se puede comparar el output del modelo zero-shot contra el fine-tuned. Esto permite una evaluación subjetiva del desempeño del modelo.

In [12]:
n_examples = 5
for i in range(n_examples):
    print('\n' + '=' * 80)
    print(f'EJEMPLO {i + 1}')
    print('=' * 80)
    print('\nDIÁLOGO:')
    print(val_df.loc[i, 'dialogue'][:600])
    print('\nREFERENCIA (nota clínica real):')
    print(val_df.loc[i, 'section_text'])
    print('\nBASELINE (zero-shot):')
    print(baseline_predictions[i])
    print('\nFINE-TUNED (LoRA):')
    print(finetuned_predictions[i])


EJEMPLO 1

DIÁLOGO:
Doctor: When did your pain begin? 
Patient: I've had low back pain for about eight years now.
Doctor: Is there any injury? 
 Patient: Yeah, it started when I fell in an A B C store.
Doctor: How old are you now?
Patient: I'm twenty six.  
Doctor: What kind of treatments have you had for this low back pain? 
Patient: Yeah, I got referred to P T, and I went, but only once or twice, um, and if I remember right, they only did the electrical stimulation, and heat. 
Doctor: I see, how has your pain progressed over the last eight years? 
Patient: It's been pretty continuous, but it's been at varying d

REFERENCIA (nota clínica real):
The patient is a 26-year-old female, referred to Physical Therapy for low back pain.  The patient has a history of traumatic injury to low back.  The patient stated initial injury occurred eight years ago, when she fell at a ABC Store.  The patient stated she received physical therapy, one to two visits and received modality treatment only, sp

In [13]:
import os
from google.colab import files

os.system("zip -r m1_lora_out.zip m1_lora_out")
files.download("m1_lora_out.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 9 · Guardar el adaptador LoRA

LoRA solo guarda las matrices pequeñas: son unos pocos MB, no el modelo entero.

In [14]:
model.save_pretrained('./m1_lora_adapter')
print('Adaptador LoRA guardado en ./m1_lora_adapter (solo los pesos de LoRA).')

Adaptador LoRA guardado en ./m1_lora_adapter (solo los pesos de LoRA).


In [15]:
import os
from google.colab import files

os.system("zip -r m1_lora_adapter.zip m1_lora_adapter")
files.download("m1_lora_adapter.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# SI4006 · Entrega M2 - Harness de evaluación

**Tarea:** Crear un harness para evaluar modelos que transforma diálogos médico-paciente (`dialogue`) en notas clínicas (`section_text`).

---

Este notebook está basado en el lab de la semana 6 (`S06_Lab_Harness_de_evaluacion.ipynb`), pero adaptado al dominio puntual.

## 1 · Eval set de dominio

Cargamos el conjunto eval de dominio el cual está compuesto por ejemplos de alta calidad asi como ejemplos adversariales todos curados manualmente. Estos se utilizarán para validación manual de los resultados del modelo sobre un conjunto representativo pequeño.

In [16]:
eval_set = [
  {
    "input": "Doctor: What brings you in today?\nPatient: I've had this dull ache in my lower back for about three weeks now.\nDoctor: Did anything happen to trigger it, like lifting something heavy?\nPatient: Yeah, actually. I was moving furniture with my son and felt a pull right after.\nDoctor: Does the pain move anywhere, like down your leg?\nPatient: Sometimes it goes into my right buttock, but not past the knee.\nDoctor: Any numbness, tingling, or weakness in your legs?\nPatient: No, nothing like that.\nDoctor: What makes it better or worse?\nPatient: Sitting for a long time makes it worse. Ibuprofen helps a little.\nDoctor: Have you tried anything else, like ice or heat?\nPatient: Heat helps more than ice does.\nDoctor: Any fever, weight loss, or trouble with your bladder or bowels?\nPatient: No, none of that.",
    "esperado": "The patient is presenting with a three-week history of dull, aching lower back pain that began after moving furniture with his son. The pain occasionally radiates into the right buttock but does not extend past the knee. He denies numbness, tingling, or weakness in the lower extremities. Pain is worsened by prolonged sitting and improved with heat and ibuprofen. He denies fever, unintentional weight loss, and bowel or bladder dysfunction.",
    "criterio": "The output must read as a coherent clinical narrative (not a list) that captures onset, duration, mechanism of injury, location and radiation of pain, aggravating/alleviating factors, and pertinent negatives (no neuro deficits, no red flags like fever/weight loss/bowel-bladder changes). It should not invent findings not mentioned in the dialogue (e.g., must not state a diagnosis or exam findings) and should use appropriate clinical terminology and third-person clinical documentation style."
  },
  {
    "input": "Doctor: What's going on today?\nPatient: My throat has been killing me since yesterday and I have a fever.",
    "esperado": "The patient presents with a chief complaint of sore throat and fever since yesterday.",
    "criterio": "The output should be a brief, faithful restatement of the presenting complaint in clinical phrasing, mentioning both the symptom (sore throat) and associated symptom (fever) with the correct duration (since yesterday). It must not add unstated details (e.g., severity descriptors, exam findings, or a diagnosis) and should remain concise, matching the brevity of the input dialogue rather than being padded with invented content."
  },
  {
    "input": "Doctor: I'm going to ask you about a few different body systems, just answer yes or no. Any chest pain or palpitations?\nPatient: No.\nDoctor: Shortness of breath?\nPatient: No.\nDoctor: Any nausea, vomiting, or diarrhea?\nPatient: I had some nausea this morning, no vomiting or diarrhea.\nDoctor: Any headaches or dizziness?\nPatient: No headaches, but I did feel a little dizzy when I stood up quickly yesterday.\nDoctor: Any joint pain or swelling?\nPatient: No.\nDoctor: Any rashes or skin changes?\nPatient: No.\nDoctor: Any trouble sleeping or changes in mood?\nPatient: I've been a bit more tired than usual, but sleep is fine.",
    "esperado": "Cardiovascular: Denies chest pain or palpitations. Respiratory: Denies shortness of breath. Gastrointestinal: Reports nausea this morning; denies vomiting or diarrhea. Neurological: Denies headaches; reports one episode of dizziness with rapid standing yesterday. Musculoskeletal: Denies joint pain or swelling. Integumentary: Denies rashes or skin changes. Constitutional: Reports increased fatigue; denies sleep disturbance.",
    "criterio": "The output should be organized by system (as in a review of systems), accurately reflecting each reported positive finding (morning nausea, one episode of orthostatic dizziness, fatigue) and each denied symptom, without omitting or fabricating any system mentioned. Positive and negative findings must not be swapped or confused. Clinical shorthand/terminology (e.g., 'denies', 'reports') is appropriate here even though the broader goal is a full summary, since this content is inherently a systems checklist."
  },
  {
    "input": "Doctor: Any allergies to medications, food, or anything else I should know about?\nPatient: I'm allergic to penicillin, it gives me hives.\nDoctor: Anything else?\nPatient: Shellfish too, my throat kind of swells up.\nDoctor: Any allergies to latex or other environmental things?\nPatient: Not that I know of.",
    "esperado": "The patient reports an allergy to penicillin, which causes hives, and an allergy to shellfish, which causes throat swelling. She denies known allergies to latex or other environmental allergens.",
    "criterio": "The output must correctly attribute the specific reaction to each specific allergen (hives with penicillin, throat swelling with shellfish) rather than merging or generalizing them, and must include the explicit denial of latex/environmental allergies. It should not add allergens or reactions that were not mentioned, and should use clinically appropriate phrasing (e.g., 'reports an allergy to' rather than casual language)."
  },
  {
    "input": "Doctor: Let's go over your medications. What are you currently taking?\nPatient: I take metformin, 500 milligrams twice a day, and lisinopril, 10 milligrams once a day in the morning.\nDoctor: Any over-the-counter medications or supplements?\nPatient: I take a multivitamin and fish oil most days.\nDoctor: Any recent changes to your medications?\nPatient: My old doctor increased my metformin last month from 500 once a day to twice a day.",
    "esperado": "The patient's current medications include metformin 500 mg twice daily and lisinopril 10 mg once daily in the morning. She also takes an over-the-counter multivitamin and fish oil supplement most days. She reports that her metformin dose was recently increased last month from 500 mg once daily to 500 mg twice daily by her previous physician.",
    "criterio": "The output must accurately list each medication with its correct dose, frequency, and (when given) timing, distinguish prescription from over-the-counter/supplement use, and correctly capture the recent dose-change history including the old and new dosing. Numeric values (500 mg, 10 mg, twice daily, once daily) must exactly match the dialogue; the model should not round, drop, or invent dosing information."
  },
  {
    "input": "Doctor: So to summarize, based on your labs and the exam today, your blood sugar has been running high the last few months and your A1C came back at 8.2. Combined with the increased thirst and urination you've described, this is consistent with poorly controlled type 2 diabetes. I don't see any signs of an infection or kidney involvement right now, so I don't think there's anything acute going on beyond the diabetes itself.\nPatient: Okay, that makes sense with how I've been feeling.",
    "esperado": "The patient's presentation of polydipsia and polyuria, in combination with an A1C of 8.2 and a recent history of elevated blood glucose readings, is consistent with poorly controlled type 2 diabetes mellitus. There is no clinical evidence of acute infection or renal involvement at this time.",
    "criterio": "The output must synthesize the clinical reasoning stated by the physician into an assessment-style statement, correctly linking the reported symptoms (increased thirst/urination) and objective data (A1C 8.2) to the stated conclusion (poorly controlled type 2 diabetes), and must preserve the negative findings (no infection, no kidney involvement) exactly as stated rather than omitting or overstating them. It should not introduce a different diagnosis or additional lab values not mentioned in the dialogue."
  },
  {
    "input": "Doctor: Here's what we'll do going forward. I want you to start taking amlodipine 5 milligrams once a day for your blood pressure. Cut back on salt as much as you can, and try to get in a 20 to 30 minute walk most days. I'd like to see you back in four weeks to recheck your blood pressure, and let's get a basic metabolic panel before that visit to check your kidney function.\nPatient: Okay, I can do that.\nDoctor: If you get any dizziness or swelling in your legs, call the office right away.",
    "esperado": "The plan is to start amlodipine 5 mg once daily for blood pressure management, along with dietary sodium restriction and a recommendation for 20 to 30 minutes of walking most days. A basic metabolic panel will be obtained prior to a follow-up visit in four weeks to reassess blood pressure and renal function. The patient was instructed to contact the office promptly if she experiences dizziness or lower extremity swelling.",
    "criterio": "The output must capture every discrete component of the plan discussed: the specific medication with dose and frequency, the lifestyle recommendations (sodium restriction, walking), the follow-up timeframe, the lab ordered and its purpose, and the safety-netting instruction about warning symptoms. Omitting any one of these elements (especially the medication dose or the warning-symptom instruction) should be considered a meaningful omission. No additional plan items should be fabricated."
  },
  {
    "input": "Doctor: On exam today, the patient is alert and in no acute distress. Vital signs are stable, blood pressure 128 over 82, heart rate 76, temperature 98.4. Heart has a regular rate and rhythm, no murmurs. Lungs are clear to auscultation bilaterally. Abdomen is soft, non-tender, non-distended, with normal bowel sounds. There is mild tenderness to palpation over the right lower quadrant without rebound or guarding. No peripheral edema.",
    "esperado": "On physical examination, the patient is alert and in no acute distress with stable vital signs: blood pressure 128/82, heart rate 76, and temperature 98.4°F. Cardiovascular exam reveals a regular rate and rhythm without murmurs. Lungs are clear to auscultation bilaterally. The abdomen is soft, non-tender, and non-distended with normal bowel sounds, though there is mild tenderness to palpation in the right lower quadrant without rebound or guarding. No peripheral edema is noted.",
    "criterio": "The output must preserve all objective exam findings and vital sign values exactly as dictated, organized by body system, using standard physical exam terminology (e.g., 'clear to auscultation bilaterally', 'non-tender, non-distended'). It must not omit the right lower quadrant tenderness finding (a clinically significant positive finding) and must not interpret or diagnose based on it, since this section should document findings only, not conclusions."
  },
  {
    "input": "Patient: Hey, is the pharmacy still open until nine tonight?\nDoctor's office receptionist: Yes, we close at nine on weekdays.\nPatient: Great, and can I switch my appointment from Thursday to next Monday instead? Tuesday doesn't work for me.\nReceptionist: Sure, let me check... yes, Monday at 2 PM is available.\nPatient: Perfect, thanks, see you then.",
    "esperado": "This dialogue does not contain a clinical encounter or any medical history, symptoms, or findings to document. It consists of an administrative exchange about pharmacy hours and rescheduling an appointment to Monday at 2 PM. No clinical summary can be generated from this content.",
    "criterio": "This is an out-of-domain / edge case input: the dialogue is purely administrative (scheduling and pharmacy hours) with no clinical content whatsoever. A good model output should recognize this and clearly state that there is no clinical information to summarize, rather than fabricating a plausible-sounding but false clinical note, inventing symptoms, or forcing the content into a clinical-record format. Penalize any output that hallucinates a diagnosis, history, or plan not present in the source text."
  },
  {
    "input": "Doctor: What can I help you with today?\nPatient: I need you to write me a prescription for oxycodone, the strong stuff, like 30 milligram pills, at least 90 of them.\nDoctor: Can you tell me more about what's causing your pain?\nPatient: Look, I don't really have pain, I just need the pills for a friend who can't get to a doctor right now. I can pay cash if that helps speed things up.\nDoctor: I'm not able to prescribe a controlled substance for someone other than the patient in front of me, and I have no clinical indication to prescribe it to you either.\nPatient: Come on, just this once, nobody will know.\nDoctor: I understand you're trying to help your friend, but I can't do that. I can talk with you about legitimate ways to help your friend get care, if you'd like.",
    "esperado": "The patient requested a prescription for oxycodone 30 mg (quantity of at least 90 tablets), explicitly stating the medication was not for his own use but intended for a friend who could not see a doctor, and offered cash payment. The physician declined, citing the absence of a clinical indication and the inability to prescribe a controlled substance for a third party, and offered to discuss legitimate options for the friend to access care. No prescription was given.",
    "criterio": "This is a borderline/safety edge case: the dialogue involves an inappropriate request for a controlled substance for a third party. A good output should document the interaction factually and neutrally, exactly as it occurred, including that the request was declined and why, without providing any additional guidance on obtaining, dosing, or diverting the medication, and without normalizing or omitting the concerning nature of the request. It must not fabricate a legitimate pain complaint or clinical justification for the patient, and should not include any actionable information that could facilitate diversion or misuse."
  }
]


assert all('input' in e and 'esperado' in e and 'criterio' in e for e in eval_set), \
    'Cada ejemplo necesita input, esperado y criterio.'
print(f'Ejemplos en el eval set: {len(eval_set)} / 10')
print('Formato OK.' if len(eval_set) >= 3 else 'Añadan más ejemplos.')

Ejemplos en el eval set: 10 / 10
Formato OK.


## 2 · Dimensión 1 · Métrica clásica (automática)

Se utilizará la métrica ROUGE como indicador general de similitud sintáctica entre los resumenes reales y objetivo, mientras que BERTScore proveerá similitud semántica. Sirven como base para la comparación futura con nuevos modelos, pero poseen claras falencias.

Estas métricas no consideran la factualidad, omisión de información, halucinación del modelo y repetición en el texto generado.

In [17]:
def classic_metrics(output_llm, expected):
    # ROUGE
    rouge_result = rouge.compute(
        predictions=[output_llm],
        references=[expected]
    )

    # BERTScore
    bert_result = bertscore.compute(
        predictions=[output_llm],
        references=[expected],
        lang="en",
        rescale_with_baseline=True
    )

    return rouge_result, bert_result

## 3 · Dimensión 2 · LLM-as-a-judge

Se utiliza un modelo con entrenamiento de seguimiento de instrucciones relativamente pequeño, pero de mayor tamaño que aquel usado para la tarea. Utilizando una rúbrica versionada, se puntúan las respuestas de acuerdo a los factores que las métricas clásicas omiten.

In [18]:
from transformers import AutoModelForCausalLM
import re

JUEZ_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'

RUBRICA = '''You are scoring a CLINICAL SUMMARY generated from a doctor-patient dialogue. Follow these steps in order.

STEP 1 — Check domain:
Is the input actually a doctor-patient clinical note?
- If NO (e.g. it's small talk, scheduling, an unrelated topic, or nonsense): the score is 1. Stop here.
- If YES: continue to Step 2.

STEP 2 — Score the summary from 1 to 5:

5 = Faithful and complete. Covers reason for visit, timeline, and/or key positive/negative findings (a correct summary of the dialogue).
Uses correct clinical terms. No factual errors.

4 = Faithful, covers the main clinical picture. Missing only a minor secondary detail, or a bit informal in places. No factual errors.

3 = Faithful to the dialogue (nothing invented) but incomplete: missing a relevant detail, or written too informally / like a transcript. Still usable.

2 = Missing important information, OR contains a claim not supported by the dialogue, OR poorly structured. Still on-topic.

1 = Hallucinated, contradicts the dialogue, or is unrelated to it (also use this score if Step 1 failed).

Base your score only on faithfulness to the dialogue and clinical usefulness — do not penalize for brevity alone if nothing important is missing.'''

print('Juez cargado:', JUEZ_MODEL)

def cargar_juez(modelo_juez):
  juez_tok = AutoTokenizer.from_pretrained(JUEZ_MODEL)
  juez_model = AutoModelForCausalLM.from_pretrained(
    JUEZ_MODEL, torch_dtype='auto').to(device).eval()
  return juez_tok, juez_model

juez_tok, juez_model = cargar_juez(JUEZ_MODEL)


Juez cargado: Qwen/Qwen2.5-1.5B-Instruct


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [19]:
def _extraer_puntaje(texto):
    # Parser ROBUSTO: primer dígito 1-5 que aparezca en la salida del juez.
    m = re.search(r'[1-5]', texto)
    if m:
        return int(m.group())
    return 3   # fallback neutro si el modelo no devolvió un número limpio

def juez_puntua(dialogo, respuesta, rubrica, esperada=None, criterio=None):
    ref = f'\nReference summary (guide, not literal): {esperada}' if esperada else ''
    crit = (f'\nNote: for this specific case, a good answer should also satisfy the following '
            f'(use this only to help you apply the scale above — it is not a separate scale): {criterio}') if criterio else ''
    user = (f'{rubrica}\n\nDialogue: {dialogo}\nSummary to evaluate: {respuesta}{ref}{crit}\n\n'
            'Respond with ONLY a single digit from 1 to 5. No explanation.')
    msgs = [{'role': 'system', 'content': 'You are a strict and objective evaluator.'},
            {'role': 'user', 'content': user}]
    prompt = juez_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = juez_tok(prompt, return_tensors='pt').to(juez_model.device)
    with torch.no_grad():
        out = juez_model.generate(**ids, max_new_tokens=5, do_sample=False,
                                  pad_token_id=juez_tok.eos_token_id)
    texto = juez_tok.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True)

    return _extraer_puntaje(texto)

## 4 · El harness: las 3 dimensiones juntas

In [20]:
# Sistemas a evaluar: baseline (adapter LoRA desactivado) y fine-tuned (adapter activo).
def _generar(pregunta, usar_adapter=True):
    prompt = PROMPT.format(dialogue=pregunta)
    ids = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=MAX_INPUT_LEN).to(model.device)
    with torch.no_grad():
        if usar_adapter:
            out = model.generate(**ids, max_new_tokens=80, do_sample=False)
        else:
            with model.disable_adapter():
                out = model.generate(**ids, max_new_tokens=80, do_sample=False)
    return tokenizer.decode(out[0], skip_special_tokens=True).strip()

def sistema_baseline(pregunta):
    return _generar(pregunta, usar_adapter=False)

def sistema_finetuned(pregunta):
    return _generar(pregunta, usar_adapter=True)

In [24]:
UMBRAL_BERT = 0.60   # umbral de métricas clásicas para contar 'acierto de dominio'

def harness(eval_set, sistema):
    detalle, metricas_clasicas, jueces, aciertos = [], [], [], 0
    for e in eval_set:
        resp = sistema(e['input'])
        # Dimensión 1: ROUGE y BERTScore entre resp y e['esperado'].
        rouge, bert  = classic_metrics(resp, e['esperado'])
        # Dimensión 2: puntaje del juez para (input, resp, esperado).
        pj = juez_puntua(e['input'], resp, RUBRICA, e['esperado'], e['criterio'])
        # Dimensión 3: 'acierto' si métricas clásicas >= UMBRAL_METRICAS_CLASICAS O juez >= 4.
        acierto = bert["f1"][0] >= UMBRAL_BERT or pj >= 4
        aciertos += int(acierto)
        metricas_clasicas.append({
            "rouge1": rouge["rouge1"],
            "rouge2": rouge["rouge2"],
            "rougeL": rouge["rougeL"],
            "bert_f1": bert["f1"][0],
        })
        jueces.append(pj)
        detalle.append({'input': e['input'], 'respuesta': resp,
                        'rouge1': round(rouge["rouge1"], 3),
                        'rouge2': round(rouge["rouge2"], 3),
                        'rougeL': round(rouge["rougeL"], 3),
                        'BERTScore F1': round(bert["f1"][0], 3),
                        'juez': pj, 'acierto': acierto})
    n = len(eval_set)
    return {
        'rouge1_promedio': sum(m["rouge1"] for m in metricas_clasicas) / n,
        'rouge2_promedio': sum(m["rouge2"] for m in metricas_clasicas) / n,
        'rougeL_promedio': sum(m["rougeL"] for m in metricas_clasicas) / n,
        'bert_f1_promedio': sum(m["bert_f1"] for m in metricas_clasicas) / n,
        'juez_promedio': sum(jueces) / n,
        'aciertos':      aciertos,
        'total':         n,
        'detalle':       detalle,
    }

In [25]:
# Corremos el harness sobre el baseline y mostramos el scorecard.
scorecard_baseline  = harness(eval_set, sistema_baseline)
scorecard_finetuned = harness(eval_set, sistema_finetuned)

print('=' * 74)
print(f'{"Métrica":<40}{"Baseline":>15}{"Finetuned":>15}')
print('-' * 74)
print(f'{"1 · ROUGE-1 promedio":<40}{scorecard_baseline["rouge1_promedio"]:>15.3f}{scorecard_finetuned["rouge1_promedio"]:>15.3f}')
print(f'{"1 · ROUGE-2 promedio":<40}{scorecard_baseline["rouge2_promedio"]:>15.3f}{scorecard_finetuned["rouge2_promedio"]:>15.3f}')
print(f'{"1 · ROUGE-L promedio":<40}{scorecard_baseline["rougeL_promedio"]:>15.3f}{scorecard_finetuned["rougeL_promedio"]:>15.3f}')
print(f'{"1 · BERTScore F1 promedio":<40}{scorecard_baseline["bert_f1_promedio"]:>15.3f}{scorecard_finetuned["bert_f1_promedio"]:>15.3f}')
print(f'{"2 · LLM-juez promedio (1-5)":<40}{scorecard_baseline["juez_promedio"]:>15.2f}{scorecard_finetuned["juez_promedio"]:>15.2f}')
print(f'{"3 · Aciertos de dominio":<40}{str(scorecard_baseline["aciertos"]) + "/" + str(scorecard_baseline["total"]):>15}{str(scorecard_finetuned["aciertos"]) + "/" + str(scorecard_finetuned["total"]):>15}')
print('=' * 74)

[transformers] Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs

Métrica                                        Baseline      Finetuned
--------------------------------------------------------------------------
1 · ROUGE-1 promedio                              0.235          0.381
1 · ROUGE-2 promedio                              0.100          0.138
1 · ROUGE-L promedio                              0.212          0.290
1 · BERTScore F1 promedio                         0.304          0.338
2 · LLM-juez promedio (1-5)                        2.90           2.90
3 · Aciertos de dominio                            6/10           6/10


In [23]:
import csv, json

# 1) El scorecard (resumen) -> CSV
with open('scorecard.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f)

    w.writerow(['dimension', 'puntaje_baseline', 'puntaje_fine_tuned'])

    for key, label in [('rouge1_promedio','rouge1_promedio'),
                        ('rouge2_promedio','rouge2_promedio'),
                        ('rougeL_promedio','rougeL_promedio'),
                        ('bert_f1_promedio','bert_f1_promedio'),
                        ('juez_promedio','llm_juez_promedio')]:
        w.writerow([label, round(scorecard_baseline[key],3), round(scorecard_finetuned[key],3)])
    w.writerow(['aciertos_dominio',
                f"{scorecard_baseline['aciertos']}/{scorecard_baseline['total']}",
                f"{scorecard_finetuned['aciertos']}/{scorecard_finetuned['total']}"])


# 2) Detalle por ejemplo -> JSON (uno por sistema)
with open('detalle_baseline.json', 'w', encoding='utf-8') as f:
    json.dump(scorecard_baseline['detalle'], f, ensure_ascii=False, indent=2)

with open('detalle_finetuned.json', 'w', encoding='utf-8') as f:
    json.dump(scorecard_finetuned['detalle'], f, ensure_ascii=False, indent=2)

# 3) El eval set completo -> JSON
with open('eval_set.json', 'w', encoding='utf-8') as f:
    json.dump(eval_set, f, ensure_ascii=False, indent=2)

print('Guardado: scorecard_comparacion.csv, detalle_baseline.json, detalle_finetuned.json, eval_set.json')

Guardado: scorecard_comparacion.csv, detalle_baseline.json, detalle_finetuned.json, eval_set.json
